# Imports/Settings

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
# 1. Стандартная библиотека
import os
import sys
import warnings
from pathlib import Path

# --- ДИНАМИЧЕСКИЙ РАСЧЕТ АБСОЛЮТНЫХ ПУТЕЙ ---
notebook_dir = Path(os.getcwd()).resolve()
if notebook_dir.name == "notebooks":
    PROJECT_ROOT = notebook_dir.parent
else:
    PROJECT_ROOT = notebook_dir

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))


# 2. Сторонние библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_classif
import sweetviz as sv

# 3. Локальные модули
from core.data import UniversalDataLoader
from core.features import TabularPreprocessor, FeatureEngineer
from core.utils import load_hydra_config, collapse_rare_categories

# --- РАБОТА С ПРЕДУПРЕЖДЕНИЯМИ ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- ИНИЦИАЛИЗАЦИЯ HYDRA ---
load_hydra_config.cache_clear()
# --- Инициализация Гидры ---
cfg = load_hydra_config()

print(f"Проект: {cfg.project_name} | Режим: Feature Engineering")
print(f"Проверка sample_pct: {cfg.data.sample_pct}")


reports_dir = Path(cfg.paths.reports_dir)
reports_dir.mkdir(parents=True, exist_ok=True)

# --- НАСТРОЙКИ ВИЗУАЛИЗАЦИИ ---
try:
    p_cfg = cfg.logging.plots
    plt.style.use(p_cfg.style)
    plt.rcParams.update({
        'figure.figsize': p_cfg.fig_size,
        'figure.dpi': p_cfg.dpi,
        'font.size': p_cfg.font_size,
        'axes.grid': p_cfg.grid,
        'axes.spines.top': p_cfg.spines_top,
        'axes.spines.right': p_cfg.spines_right
    })
    PLOT_ALPHA = p_cfg.alpha
except AttributeError:
    PLOT_ALPHA = 0.5
    print("Используются дефолтные стили Matplotlib.")

Проект: outsource_project_name | Режим: Feature Engineering
Проверка sample_pct: 0.05


# Data Loading

In [3]:
loader = UniversalDataLoader(cfg)

df = loader.load_data()
target = cfg.data.tabular.target_col

print(f"Размер датасета: {df.shape}")
df.head(3)

Размер датасета: (93879, 30)


,session_id,client_id,visit_date,visit_time,visit_number,utm_source,utm_medium,utm_campaign,utm_adcontent,utm_keyword,...,first_hit_time_ms,last_hit_time_ms,total_events_count,unique_event_actions,unique_cars_viewed,total_car_views,is_first_hit_car_view,event_value,hits_before_target,car_view_ratio
0,1000302132601256598.1640200867.1640200867,232900989.1640201,2021-12-22,22:21:07,1,ZpYIoDJMcFzVoPFsHGJL,banner,LEoPHuyFvzoNfnzGgfcd,vCIpmpaGBnIQhyYNkXqp,puhZPIYqKXeFPaUviSjo,...,952.0,16392.0,3,1,0,0,0,0,3,0.0
1,1000593202533074166.1638368503.1638368503,232968759.1638368502,2021-12-01,17:21:43,1,MvfHsxITijuriZxsqZqt,cpm,FTjNLDyTrXaWYgZymFkV,xhoenQgDQsgfEPYNPwKO,RrhnkuoaqckNtJpAZDzH,...,1200.0,1200.0,1,1,0,0,0,0,1,0.0
2,1000739068214098694.1640086546.1640086546,233002721.1640086278,2021-12-21,14:35:46,1,MvfHsxITijuriZxsqZqt,cpm,FTjNLDyTrXaWYgZymFkV,xhoenQgDQsgfEPYNPwKO,MWLEpQPyjGkjHseVyeyQ,...,0.0,0.0,0,0,0,0,0,0,0,0.0


In [4]:
df.to_parquet(f"{cfg.paths.features_dir}/df_aggregated.pq", index=False)

In [17]:
train_df, val_df, test_df = loader.get_splits(df)

target_col = cfg.data.tabular.target_col # 'event_value'
y_train = train_df[target_col].values
y_val = val_df[target_col].values

In [18]:
X_train_raw = train_df.drop(columns=[target_col])
X_val_raw = val_df.drop(columns=[target_col])

# Feature Generation

In [22]:
engineer = FeatureEngineer(cfg)

X_train_engineered = engineer.fit_transform(X_train_raw)

In [23]:
preprocessor = TabularPreprocessor(cfg)

X_train = preprocessor.fit_transform(X_train_engineered, y_train)

In [24]:
eda_df = X_train.copy()
eda_df[target_col] = y_train

# Схлопываем редкие категории, чтобы Sweetviz построил красивые графики без каши
cat_cols = eda_df.select_dtypes(exclude=[np.number]).columns.tolist()
eda_df_collapsed = collapse_rare_categories(eda_df, cat_cols, top_n=12)

feature_config = sv.FeatureConfig()
feature_config.force_num = [target_col]
feature_config.force_cat = ['day_of_week', 'is_weekend', 'is_night']

report = sv.analyze(eda_df_collapsed, target_feat=target_col, feat_cfg=feature_config)
output_path = reports_dir / "eda_featuredv2.0.html"
report.show_html(filepath=str(output_path), open_browser=True)

                                             |          | [  0%]   00:00 -> (? left)

Report reports\eda_featuredv2.0.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.
